# 13 — Unexpected credit contraction (fixed FX, leverage 1/16)

**Ports**: `Code/MATLAB/Transition/solve_transition_fixed.m` and `Main.m` lines 360–381 (the fixed-FX block). Reuses `backsolve_egm` from notebook 10/12 with the same parameterisation; adds a NKPC backward iteration to back out marginal cost, real wage, MPK, and the real interest rate path.

**Why a separate solver**: under fixed nominal exchange rate, terms of trade can only move via *domestic* price changes, which are penalised by the Rotemberg adjustment cost ($\theta_{nr} = 100$). So `tot` becomes an *explicit unknown* (no per-period 1-D solve as in flex); the residual system gains one block (goods-market clearing).

**Unknowns**: $XX = [k(2..TT);\, l(1..TT);\, \text{tot}(1..TT)]$ of length $3\,TT-1 = 599$.

**Residuals** (length $3\,TT-1$):
1. capital: $k'(t) - k_\text{agg}(t+1) = 0$ for $t=1..TT-1$
2. labour-Euler: $l'_\text{prime}(t) - \text{lab\_agg}(t) = 0$ for $t=1..TT$
3. goods-market: $\text{net\_c}(t) - c_t(t) = 0$ for $t=1..TT$

**NKPC backward block** (`solve_transition_fixed.m` lines 35–56):
$$\text{mc}_t = \frac{\varepsilon_f - 1}{\varepsilon_f} + \frac{\theta_{nr}}{\varepsilon_f}(\pi_t - 1)\pi_t - \frac{1}{1+r_{t+1}}\frac{\theta_{nr}}{\varepsilon_f}(\pi_{t+1} - 1)\pi_{t+1}\frac{y_{t+1}}{y_t},$$
where $\pi_t = \text{tot}_t/\text{tot}_{t-1}$ and $\pi_0 = \text{tot}_0/\text{tot}_\text{initial}$. Then $w_t,\,\text{MPK}_t$ follow from cost minimisation and $r_t = (\text{MPK}_t + (1-\delta)q_t)/q_{t-1} - 1$.

**Faithful port note**: `solve_transition_fixed.m` line 65 references `k_agg(t)` in the adjustment-cost term where `t` is the leftover loop variable (= 1, after the for-loop exits). This is almost certainly a MATLAB bug. We replicate it (with a clear comment) so that residuals match MATLAB's. We also fix MATLAB's double-negative `- -theta_nr/2*(t_inf-1).^2` which simplifies to `+ theta_nr/2*(t_inf-1).^2`.

**Strategy**:
1. Load the converged flex contraction (notebook 12) as warm-start: $XX_0 = [k_\text{flex}(2..TT);\, l_\text{flex};\, \text{tot}_\text{flex}]$.
2. Use Levenberg–Marquardt (`scipy.optimize.root(method='lm')`) which is more robust than the default trust-region for this problem.
3. Save outputs for notebook 14.

**Runtime**: ~20 minutes single-core (~5 LM iterations × ~600 finite-difference Jacobian columns × ~0.4s per residual = ~1200s). Logged with `time.perf_counter()`.

## Imports + load Phase B/C/D output

In [1]:
from pathlib import Path
import time

import numpy as np
from scipy.optimize import brentq, fsolve, root
from scipy.interpolate import PchipInterpolator

OUTPUT_DIR = Path('..') / 'output'

p = np.load(OUTPUT_DIR / 'params.npz')
cb = np.load(OUTPUT_DIR / 'calibration.npz')
sh = np.load(OUTPUT_DIR / 'shock_path.npz')
ini = np.load(OUTPUT_DIR / 'initial_ss.npz')
tr = np.load(OUTPUT_DIR / 'transition_flex.npz')
contr = np.load(OUTPUT_DIR / 'contraction_flex.npz')

Agrid = p['params__Agrid']
Agrid_fine = p['params__Agrid_fine']
piex = p['params__piex']
ex = p['params__ex']
nA = int(p['params__nA'])
nA_fine = int(p['params__nA_fine'])
ns = int(p['params__ns'])
Index_b_min = int(p['params__Index_b_min'])
bmin = float(p['params__bmin'])
delta = float(p['params__delta'])
alpha = float(p['params__alpha'])
gamma = float(p['params__gamma'])
theta = float(p['params__theta'])
omega = float(p['params__omega'])
chi_dis = float(p['params__chi_dis'])
epsilon_f = float(p['params__epsilon_f'])
epsilon_w = float(p['params__epsilon_w'])
theta_nr = float(p['params__theta_nr'])
z_ss = float(p['params__z_ss'])
ex_mean = float(p['params__ex_mean'])
k_ss = float(p['params__k_ss'])
b_agg_ss = float(p['params__b_agg_ss'])

beta_calibrated = float(cb['beta_calibrated'])
cpol_calibrated = cb['cpol']
c_fine_calibrated = cb['c_fine']

shock_b_agg = sh['shock_b_agg']
new_shock_b_agg = sh['new_shock_b_agg']
TT = int(sh['TT'])
d_s = float(ini['d_s'])
theta_star = int(ini['theta_star'])
z_t = np.ones(TT) * z_ss

shock_start_idx = int(tr['shock_start_idx'])
G0_shock = tr['dist_at_shock_start']
k_initial = float(contr['k_initial'])
q_initial = float(contr['q_initial'])
tot_initial = float(contr['tot_initial'])
BG_shock = float(contr['BG_shock'])
k_final = float(contr['k_final'])

k_const_hat = 1.0 / 16.0
k_const = k_const_hat * k_initial
phi_ac = 17.0

# Warm start from flex contraction
k_l_t_fixed_init = np.concatenate([
    contr['k_agg'][1:],   # k(2..TT)
    contr['lab_agg'],      # l(1..TT)
    contr['tot'],          # tot(1..TT)
])
print(f'TT = {TT}, 3*TT-1 = {3*TT-1}, warm-start shape = {k_l_t_fixed_init.shape}')
print(f'k_initial = {k_initial:.4f}, BG_shock = {BG_shock:.4f}, k_const_hat = {k_const_hat:.4f}')
print(f'theta_nr = {theta_nr}  (Rotemberg adjustment cost on tot)')
print(f'Warm-start tot[0] = {contr["tot"][0]:.4f} (flex value, almost certainly too volatile for fixed FX)')

TT = 200, 3*TT-1 = 599, warm-start shape = (599,)
k_initial = 29.7513, BG_shock = -6.2810, k_const_hat = 0.0625
theta_nr = 100.0  (Rotemberg adjustment cost on tot)
Warm-start tot[0] = 0.9099 (flex value, almost certainly too volatile for fixed FX)


## Helpers + `backsolve_egm` (with the borrowing-constraint clip)

The MATLAB `backsolve_egm.m` raises an error if `dec_temp_fine < bmin`. Under the fixed-FX outer solver the trust-region updates can transiently produce candidate $XX$'s that imply slightly infeasible decisions; raising prevents `fsolve`/`root` from making progress. We instead **clip** to `bmin` (the standard borrowing-constraint treatment), which produces a smooth residual surface.

In [2]:
def basefun_vec(grid, x):
    n = grid.size
    ind1 = np.searchsorted(grid, x, side='right') - 1
    ind1 = np.clip(ind1, 0, n - 2)
    ind2 = ind1 + 1
    w2 = (x - grid[ind1]) / (grid[ind2] - grid[ind1])
    w2 = np.clip(w2, 0.0, 1.0)
    w1 = 1.0 - w2
    return ind1, ind2, w1, w2


def backsolve_egm(r_t, t_guess, y_agg, cpol, c_fine, prof_firm, Profit_i, G0,
                  *, k_init_arg, q_init_arg, tot_init_arg, BG_shock_arg,
                  k_const_hat_arg, k_const_arg, foreign_arg=True):
    sav_grid = np.tile(Agrid[:, None], (1, ns))
    exLw = np.tile(ex[None, :], (nA, 1))
    pos = Agrid_fine > 0
    net_a_rich = (G0[pos, :].sum(axis=1) * Agrid_fine[pos]).sum()
    if BG_shock_arg == 0.0:
        lev_rat_rich = 1.0
    else:
        lev_rat_rich = (1.0 - k_const_hat_arg) * k_init_arg * q_init_arg / max(net_a_rich, 1e-12)
    Kgrid_fine = np.maximum(lev_rat_rich * Agrid_fine + k_const_arg, 0.0)
    Bgrid_fine = -(Kgrid_fine - Agrid_fine)

    c_tran = np.full((nA, ns, TT), np.nan)
    c_tran_fine_out = np.full((nA_fine, ns, TT), np.nan)
    dec_tran = np.full((nA, ns, TT), np.nan)
    dec_tran_fine = np.full((nA_fine, ns, TT), np.nan)
    c_tran[:, :, TT - 1] = cpol
    c_tran_fine_out[:, :, TT - 1] = c_fine

    for t in range(TT - 2, -1, -1):
        r = r_t[t + 1]
        r_yesterday = r_t[t]
        tot_t = t_guess[t]
        y_t_agg = y_agg[t]
        prof_firms_t = prof_firm[t] + Profit_i[t]
        mult = 1.0 + tot_t ** (theta - 1.0) * omega / (1.0 - omega)
        c_constrained = np.maximum(
            (1.0 / mult) * (
                (1.0 + r_yesterday) * sav_grid
                + (epsilon_f - 1.0) / epsilon_f * exLw * y_t_agg * (1.0 - alpha) / ex_mean
                + prof_firms_t - bmin
            ),
            1e-5,
        )
        Emup1 = c_tran[:, :, t + 1] ** (-gamma)
        Emup = beta_calibrated * (1.0 + r) * (Emup1 @ piex.T)
        c_s = Emup ** (-1.0 / gamma)
        a_today = (
            c_s * mult + sav_grid
            - (epsilon_f - 1.0) / epsilon_f * exLw * y_t_agg * (1.0 - alpha) / ex_mean
            - prof_firms_t
        ) / (1.0 + r_yesterday)
        c_new = np.empty((nA, ns))
        c_new_fine = np.empty((nA_fine, ns))
        dec_temp_fine = np.empty((nA_fine, ns))
        for j in range(ns):
            xj = a_today[:, j]
            cj = c_s[:, j]
            threshold = a_today[Index_b_min, j]
            mask = Agrid > threshold
            interp = PchipInterpolator(xj, cj, extrapolate=True)
            c_new[:, j] = np.where(mask, interp(Agrid), c_constrained[:, j])
            c_new[:, j] = np.maximum(c_new[:, j], 1e-5)
            mask_fine = Agrid_fine > threshold
            interp_unc = PchipInterpolator(xj, cj, extrapolate=True)
            interp_con = PchipInterpolator(Agrid, c_constrained[:, j], extrapolate=True)
            c_new_fine[:, j] = np.where(
                mask_fine, interp_unc(Agrid_fine), interp_con(Agrid_fine)
            )
            dec_unc = (
                (1.0 + r_yesterday) * Agrid_fine
                + (epsilon_f - 1.0) / epsilon_f * ex[j] * y_t_agg * (1.0 - alpha) / ex_mean
                + prof_firms_t
                - c_new_fine[:, j] * mult
            )
            dec_temp_fine[:, j] = np.where(mask_fine, dec_unc, bmin)
        c_tran[:, :, t] = c_new
        dec_tran[:, :, t] = (
            (1.0 + r_yesterday) * sav_grid
            + (epsilon_f - 1.0) / epsilon_f * exLw * y_t_agg * (1.0 - alpha) / ex_mean
            + prof_firms_t - c_new * mult
        )
        # Clip below borrowing limit (smooth treatment for outer solver)
        dec_temp_fine = np.maximum(dec_temp_fine, bmin)
        c_tran_fine_out[:, :, t] = c_new_fine
        dec_tran_fine[:, :, t] = dec_temp_fine

    dist_tran = np.zeros((nA_fine, ns, TT))
    if foreign_arg:
        Bgrid_fine_new = Bgrid_fine / (t_guess[0] / tot_init_arg)
        A_grid_fine_new = Bgrid_fine_new + Kgrid_fine
        if bmin / t_guess[0] < Agrid_fine[0]:
            raise RuntimeError('positive mass would land outside Agrid_fine')
        ind1, ind2, w1, w2 = basefun_vec(Agrid_fine, A_grid_fine_new)
        G0_new = np.zeros_like(G0)
        for ai in range(nA_fine):
            for j in range(ns):
                G0_new[ind1[ai], j] += G0[ai, j] * w1[ai]
                G0_new[ind2[ai], j] += G0[ai, j] * w2[ai]
        if t_guess[0] > 1:
            G0_new = np.nan_to_num(G0_new, nan=0.0)
        if G0_new.sum() < 1.0 - 1e-6:
            raise RuntimeError('Revaluated G0 does not sum to one')
        dist_tran[:, :, 0] = G0_new
    else:
        dist_tran[:, :, 0] = G0

    a_prime_t = np.zeros(TT)
    c_t_out = np.zeros(TT)
    a_prime_t[0] = (dist_tran[:, :, 0] * dec_tran_fine[:, :, 0]).sum()
    c_t_out[0] = (dist_tran[:, :, 0] * c_tran_fine_out[:, :, 0]).sum()
    for t in range(1, TT - 1):
        prev = dist_tran[:, :, t - 1]
        ind1, ind2, w1, w2 = basefun_vec(Agrid_fine, dec_tran_fine[:, :, t - 1])
        new_dist = np.zeros((nA_fine, ns))
        for j in range(ns):
            mass = prev[:, j]
            contrib_low = mass * w1[:, j]
            contrib_hi = mass * w2[:, j]
            for jp in range(ns):
                np.add.at(new_dist[:, jp], ind1[:, j], contrib_low * piex[j, jp])
                np.add.at(new_dist[:, jp], ind2[:, j], contrib_hi * piex[j, jp])
        dist_tran[:, :, t] = new_dist
        a_prime_t[t] = (new_dist * dec_tran_fine[:, :, t]).sum()
        c_t_out[t] = (new_dist * c_tran_fine_out[:, :, t]).sum()
    return c_t_out, a_prime_t, dec_tran, dec_tran_fine, c_tran_fine_out, dist_tran[:, :, 0], dist_tran

## `solve_transition_fixed`

In [3]:
def solve_transition_fixed(XX, z_t_in, shock_b_agg_in, cpol, c_fine, G0,
                            *, k_init_arg, k_final_arg, q_init_arg, tot_init_arg,
                            BG_shock_arg, k_const_hat_arg, k_const_arg):
    k_agg = np.empty(TT); k_agg[0] = k_init_arg; k_agg[1:] = XX[:TT - 1]
    lab_agg = XX[TT - 1:2 * TT - 1]
    t_guess = XX[2 * TT - 1:]
    i_agg = k_agg[1:] - (1.0 - delta) * k_agg[:-1]
    q_agg = np.empty(TT)
    q_agg[:TT - 1] = 1.0 + phi_ac * (i_agg - delta * k_agg[:TT - 1]) / k_agg[:TT - 1]
    q_agg[TT - 1] = 1.0
    y_agg = z_t_in * (k_agg ** alpha) * (lab_agg ** (1.0 - alpha))

    a_prime = np.empty(TT)
    a_prime[:TT - 1] = shock_b_agg_in[:TT - 1] + q_agg[:TT - 1] * k_agg[1:]
    a_prime[TT - 1] = q_agg[TT - 1] * k_final_arg + shock_b_agg_in[TT - 1]
    a_t_agg = np.empty(TT); a_t_agg[0] = q_init_arg * k_agg[0] + BG_shock_arg; a_t_agg[1:] = a_prime[:-1]
    Profits_i = np.empty(TT)
    Profits_i[:TT - 1] = (
        (q_agg[:TT - 1] - 1.0) * i_agg
        - (phi_ac / 2.0) * k_agg[:TT - 1] * ((i_agg - delta * k_agg[:TT - 1]) / k_agg[:TT - 1]) ** 2
    )
    Profits_i[TT - 1] = 0.0

    t_inf = np.empty(TT)
    t_inf[0] = t_guess[0] / tot_init_arg
    t_inf[1:] = t_guess[1:] / t_guess[:-1]

    mc = np.empty(TT); w_guess = np.empty(TT); MPK_guess = np.empty(TT); r_t = np.empty(TT)
    mc[TT - 1] = (epsilon_f - 1.0) / epsilon_f + theta_nr / epsilon_f * (t_inf[TT - 1] - 1.0) * t_inf[TT - 1]
    w_guess[TT - 1] = mc[TT - 1] * (alpha ** alpha) * ((1.0 - alpha) ** (1.0 - alpha)) \
        * ((alpha / (1.0 - alpha)) ** (-alpha)) * (k_agg[TT - 1] / lab_agg[TT - 1]) ** alpha
    MPK_guess[TT - 1] = alpha * w_guess[TT - 1] * lab_agg[TT - 1] / ((1.0 - alpha) * k_agg[TT - 1])
    r_t[TT - 1] = (MPK_guess[TT - 1] + (1.0 - delta) * q_agg[TT - 1]) / q_agg[TT - 2] - 1.0
    for t in range(TT - 2, -1, -1):
        mc[t] = (epsilon_f - 1.0) / epsilon_f + theta_nr / epsilon_f * (t_inf[t] - 1.0) * t_inf[t] \
                - 1.0 / (1.0 + r_t[t + 1]) * theta_nr / epsilon_f * (t_inf[t + 1] - 1.0) * t_inf[t + 1] * y_agg[t + 1] / y_agg[t]
        w_guess[t] = mc[t] * (alpha ** alpha) * ((1.0 - alpha) ** (1.0 - alpha)) \
            * ((alpha / (1.0 - alpha)) ** (-alpha)) * (k_agg[t] / lab_agg[t]) ** alpha
        MPK_guess[t] = alpha * w_guess[t] * lab_agg[t] / ((1.0 - alpha) * k_agg[t])
        if t > 0:
            r_t[t] = (MPK_guess[t] + (1.0 - delta) * q_agg[t]) / q_agg[t - 1] - 1.0
        else:
            r_t[t] = (MPK_guess[t] + (1.0 - delta) * q_agg[t]) / q_init_arg - 1.0

    Profits_guess = y_agg - MPK_guess * k_agg - w_guess * lab_agg - theta_nr / 2.0 * (t_inf - 1.0) ** 2

    # Faithful port: MATLAB line 65 uses k_agg(t) where t = 1 (leftover loop var). Replicated.
    t_left = 0
    net_c = np.empty(TT)
    net_c[:TT - 1] = (
        y_agg[:TT - 1] - i_agg[:TT - 1]
        - t_guess[:TT - 1] ** (-theta_star) * d_s
        - (phi_ac / 2.0) * k_agg[:TT - 1] * ((i_agg[:TT - 1] - delta * k_agg[t_left]) / k_agg[t_left]) ** 2
        + theta_nr / 2.0 * (t_inf[:TT - 1] - 1.0) ** 2   # `+`, not `-` (MATLAB has `- -`)
    )
    net_c[TT - 1] = (
        y_agg[TT - 1] - i_agg[TT - 2]
        - t_guess[TT - 1] ** (-theta_star) * d_s
        - (phi_ac / 2.0) * k_agg[TT - 1] * ((i_agg[TT - 2] - delta * k_agg[TT - 1]) / k_agg[TT - 1]) ** 2
        + theta_nr / 2.0 * (t_inf[TT - 1] - 1.0) ** 2
    )

    r_k = MPK_guess; w_r = w_guess; Profits_firms = Profits_guess
    tot = t_guess.copy()
    c_t, a_prime_t, dec_tran, dec_tran_fine, c_tran_fine, In_dis, dist_tran = backsolve_egm(
        r_t, tot, y_agg, cpol, c_fine, Profits_firms, Profits_i, G0,
        k_init_arg=k_init_arg, q_init_arg=q_init_arg, tot_init_arg=tot_init_arg,
        BG_shock_arg=BG_shock_arg, k_const_hat_arg=k_const_hat_arg, k_const_arg=k_const_arg,
    )
    c_t[TT - 1] = c_t[TT - 2]
    k_prime_t = (a_prime_t[:TT - 1] - shock_b_agg_in[:TT - 1]) / q_agg[:TT - 1]
    lab_prime = (1.0 - omega) * (epsilon_w - 1.0) / epsilon_w * w_r / (c_t * chi_dis)
    resid = np.empty(3 * TT - 1)
    resid[:TT - 1] = k_prime_t - k_agg[1:]
    resid[TT - 1:2 * TT - 1] = lab_prime - lab_agg
    resid[2 * TT - 1:] = net_c - c_t
    return resid, dict(
        r_t=r_t, r_k=r_k, y_agg=y_agg, c_t=c_t, tot=tot,
        lab_agg=lab_agg, w_r=w_r, q_agg=q_agg, k_agg=k_agg,
        a_prime=a_prime, a_t_agg=a_t_agg,
        Profits=Profits_firms + Profits_i, Profits_i=Profits_i,
        Profits_firms=Profits_firms, net_c=net_c, In_dis=In_dis,
        t_inf=t_inf, mc=mc,
    )

## Time the warm-start residual

In [4]:
tic = time.perf_counter()
resid_init, _ = solve_transition_fixed(
    k_l_t_fixed_init, z_t, new_shock_b_agg, cpol_calibrated, c_fine_calibrated, G0_shock,
    k_init_arg=k_initial, k_final_arg=k_final, q_init_arg=q_initial,
    tot_init_arg=tot_initial, BG_shock_arg=BG_shock,
    k_const_hat_arg=k_const_hat, k_const_arg=k_const,
)
elapsed_one = time.perf_counter() - tic
print(f'Warm-start residual: max|r| = {np.max(np.abs(resid_init)):.3e}')
print(f'  k:   {np.max(np.abs(resid_init[:TT-1])):.3e}')
print(f'  lab: {np.max(np.abs(resid_init[TT-1:2*TT-1])):.3e}')
print(f'  c:   {np.max(np.abs(resid_init[2*TT-1:])):.3e}')
print(f'One eval: {elapsed_one:.3f}s')

Warm-start residual: max|r| = 2.792e+00
  k:   2.792e+00
  lab: 1.659e+00
  c:   5.790e-01
One eval: 0.374s


## Solve via `scipy.optimize.root(method='lm')` (timed)

In [5]:
def f_only(XX):
    r, _ = solve_transition_fixed(
        XX, z_t, new_shock_b_agg, cpol_calibrated, c_fine_calibrated, G0_shock,
        k_init_arg=k_initial, k_final_arg=k_final, q_init_arg=q_initial,
        tot_init_arg=tot_initial, BG_shock_arg=BG_shock,
        k_const_hat_arg=k_const_hat, k_const_arg=k_const,
    )
    return r

tic_total = time.perf_counter()
sol = root(
    f_only, k_l_t_fixed_init, method='lm',
    options=dict(xtol=1e-9, ftol=1e-9, maxiter=8 * 600),
)
elapsed_lm = time.perf_counter() - tic_total
XX_star = sol.x
print(f'LM root: success={sol.success}, nfev={sol.nfev}, elapsed={elapsed_lm:.1f}s ({elapsed_lm/60:.2f} min)')
print(f'  message: {sol.message}')

resid_final, out = solve_transition_fixed(
    XX_star, z_t, new_shock_b_agg, cpol_calibrated, c_fine_calibrated, G0_shock,
    k_init_arg=k_initial, k_final_arg=k_final, q_init_arg=q_initial,
    tot_init_arg=tot_initial, BG_shock_arg=BG_shock,
    k_const_hat_arg=k_const_hat, k_const_arg=k_const,
)
print(f'Final residual: max|r| = {np.max(np.abs(resid_final)):.3e}')
print(f'  k:   {np.max(np.abs(resid_final[:TT-1])):.3e}')
print(f'  lab: {np.max(np.abs(resid_final[TT-1:2*TT-1])):.3e}')
print(f'  c:   {np.max(np.abs(resid_final[2*TT-1:])):.3e}')

LM root: success=True, nfev=3004, elapsed=1185.1s (19.75 min)
  message: The relative error between two consecutive iterates is at most 0.000000


Final residual: max|r| = 3.872e-13
  k:   3.872e-13
  lab: 4.086e-14
  c:   4.707e-14


## Quick numerical comparison: flex vs. fixed FX

In [6]:
print('              flex            fixed FX')
print('-' * 50)
for name, key in [('k_agg[0]', 'k_agg'), ('k_agg[1]', 'k_agg'),
                  ('lab_agg[0]', 'lab_agg'), ('tot[0]', 'tot'),
                  ('c_t[0]', 'c_t'), ('r_t[0] (%)', 'r_t'), ('w_r[0]', 'w_r')]:
    idx = 1 if '[1]' in name else 0
    fl = contr[key][idx]
    fi = out[key][idx]
    if 'r_t' in name:
        fl = 100 * fl; fi = 100 * fi
    print(f'  {name:<14s}{fl:10.4f}      {fi:10.4f}')

print()
print(f'=== Phase D timings (notebook 13) ===')
print(f'  one residual eval     = {elapsed_one:.3f}s')
print(f'  Levenberg–Marquardt   = {elapsed_lm:.1f}s ({elapsed_lm/60:.2f} min)')

              flex            fixed FX
--------------------------------------------------
  k_agg[0]         29.7513         29.7513
  k_agg[1]         29.6723         29.7608
  lab_agg[0]        1.0656          0.9781
  tot[0]            0.9099          1.0142
  c_t[0]            1.3361          1.4477
  r_t[0] (%)       -4.8649         -0.2629
  w_r[0]            1.8091          1.7993

=== Phase D timings (notebook 13) ===
  one residual eval     = 0.374s
  Levenberg–Marquardt   = 1185.1s (19.75 min)


## Save outputs

In [7]:
out_path = OUTPUT_DIR / 'contraction_fixed.npz'
np.savez(
    out_path,
    XX_star=XX_star, resid_final=resid_final,
    elapsed_seconds=elapsed_lm, nfev=sol.nfev,
    k_const_hat=k_const_hat, k_const=k_const,
    k_initial=k_initial, q_initial=q_initial, tot_initial=tot_initial,
    BG_shock=BG_shock, k_final=k_final,
    k_agg=out['k_agg'], lab_agg=out['lab_agg'], y_agg=out['y_agg'],
    tot=out['tot'], c_t=out['c_t'],
    r_t=out['r_t'], r_k=out['r_k'], w_r=out['w_r'], q_agg=out['q_agg'],
    a_prime=out['a_prime'], a_t_agg=out['a_t_agg'],
    Profits=out['Profits'], Profits_i=out['Profits_i'],
    Profits_firms=out['Profits_firms'], net_c=out['net_c'],
    In_dis=out['In_dis'], t_inf=out['t_inf'], mc=out['mc'],
)
print(f'Saved: {out_path.resolve()}')

Saved: /Users/siyingli/github/de-Ferra2020-kz/Code/Python/output/contraction_fixed.npz
